Empezamos a practicar:


In [1]:
import oracledb
import pandas as pd
try:
    conn = oracledb.connect(
        user="hr",
        password="basededatos",
        dsn="localhost:1521/XE"  
    )
    cursor = conn.cursor()
    cursor.callproc("dbms_output.enable")


except oracledb.Error as e:
    print(f"Error detectado: {e}")

#

In [2]:
try:
    cursor.execute(f"""
    DECLARE
        v_numero NUMBER(2) NOT NULL := 10;
        v_caracter VARCHAR2(10) := 'Hace frio';
    BEGIN
        DBMS_OUTPUT.PUT_LINE(v_caracter || ' y tengo: ' || TO_CHAR(v_numero));
    END;                   
    """)

    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())

except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

Hace frio y tengo: 10


In [3]:
# --- SIMULACIÓN DE COMANDO DEFINE ---
# En SQL*Plus harías: DEFINE p_num1 = 100
# En Python definimos variables nativas:
p_num1 = 500  # Primer número (Dividendo)
p_num2 = 5    # Segundo número (Divisor)

try:
    # Usamos f-string (la f antes de las comillas) para inyectar los valores
    # Esto cumple la función exacta del "&" en SQL*Plus
    cursor.execute(f"""
    DECLARE
        v_n1 NUMBER := {p_num1};  -- Aquí Python pega el 500
        v_n2 NUMBER := {p_num2};  -- Aquí Python pega el 5
        v_resultado NUMBER;
    BEGIN
        v_resultado := v_n1 / v_n2;
        
        DBMS_OUTPUT.PUT_LINE('El numero ' || v_n1 || ' dividido para ' || v_n2 || ' es: ' || v_resultado);
        
        -- Adicionamos el resultado a una variable (como pide el taller)
        -- y mostramos
        v_resultado := v_resultado + 100;
        DBMS_OUTPUT.PUT_LINE('Resultado + 100 = ' || v_resultado);
    END;
    """)

    # --- CODIGO DE SALIDA (Tu bloque while estándar) ---
    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())

except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

El numero 500 dividido para 5 es: 100
Resultado + 100 = 200


In [5]:
p_salary = 50000
p_bonus = 10 

try:
    cursor.execute(
    f"""
    DECLARE
        v_salary NUMBER := {p_salary};
        v_bonus NUMBER := {p_bonus};
    BEGIN
        DBMS_OUTPUT.PUT_LINE('TOTAL_COMPENSACION: ' || TO_CHAR(v_salary + (v_salary * v_bonus/100)));
    END;
    """
    )

    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())

except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

TOTAL_COMPENSACION: 55000


Crear un bloque PL/SQL que imprima información sobre el máximo salario pagado a los empleados guarde esta información en una variable y muéstrela en pantalla

In [6]:
try:
    cursor.execute(
    f"""
    DECLARE
        v_salario_maximo employees.salary%TYPE;
    BEGIN
        SELECT MAX(salary)
        INTO v_salario_maximo
        FROM employees;
        DBMS_OUTPUT.PUT_LINE('El salario maximo es: ' || TO_CHAR(v_salario_maximo));
    END;
    """
    )

    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())

except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

El salario maximo es: 32044


Crear un bloque PL/SQL para insertar un nuevo departamento en la tabla DEPARTMENTS.
a. Utilice el comando DEFINE para ingresar el código, nombre, del nuevo departamento llamado “EDUCACION”
b. Registre el código location y manager como null por esta vez
c. Muestre en pantalla el nuevo departamento creado

In [13]:
try:
    p_nombre = "EDUCACION"
    p_codigo = 12
    cursor.execute(
    f"""
    DECLARE
        v_nombre departments.department_name%TYPE := '{p_nombre}';
        v_codigo departments.department_id%TYPE := {p_codigo};
    BEGIN
        INSERT INTO departments
        (department_id, department_name)
        VALUES
        ({p_codigo}, '{p_nombre}');
        SELECT department_id, department_name
        INTO v_codigo, v_nombre
        FROM departments
        WHERE department_id = {p_codigo};
        DBMS_OUTPUT.PUT_LINE('Codigo depa: ' || TO_CHAR(v_codigo) || ', nombre depa: ' || v_nombre);
        INSERT INTO employees
        (employee_id, last_name, email, hire_date, job_id, salary)
        VALUES
        (1885, 'Cardenas', 'ac@ac', SYSDATE, 2, 15000);
        SELECT employee_id, last_name
        INTO v_codigo, v_nombre
        FROM employees
        WHERE employee_id = 1885;
        DBMS_OUTPUT.PUT_LINE('Codigo emple: ' || TO_CHAR(v_codigo) || ', nombre emple: ' || v_nombre);
        DELETE FROM employees
        WHERE employee_id = 1885;
        DELETE FROM departments
        WHERE department_id = {p_codigo};
    END;
    """
    )

    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())

except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

Codigo depa: 12, nombre depa: EDUCACION
Codigo emple: 1885, nombre emple: Cardenas


Crear un bloque PL/SQL para calcular un bono de acuerdo a lo siguiente:
a. Utilice el comando DEFINE para indicar el código del empleado
b. Si el empleado tiene un salario menor a $5000, otorgue en el bono un valor del 10% del salario del empleado
c. Si el empleado tiene un salario entre a $5001 y $10.000, otorgue en el bono un valor del 15% del salario del empleado
d. Si el empleado tiene un salario mayor a $10.001, otorgue en el bono un valor del 20% del salario del empleado

In [20]:
try:
    p_codigo = 100
    cursor.execute(
    f"""
    DECLARE
        v_codigo employees.employee_id%TYPE := {p_codigo};
        v_salario employees.salary%TYPE;
    BEGIN
        SELECT salary
        INTO v_salario
        FROM employees
        WHERE employee_id = v_codigo;
        IF v_salario < 5000
        THEN DBMS_OUTPUT.PUT_LINE('Bono del 10%: ' || TO_CHAR(v_salario * 1.1));
        ELSIF v_salario BETWEEN 5001 AND 10000
        THEN DBMS_OUTPUT.PUT_LINE('Bono del 15%: ' || TO_CHAR(v_salario * 1.15));
        ELSE DBMS_OUTPUT.PUT_LINE('Bono del 20%: ' || TO_CHAR(v_salario * 1.2));
        END IF;
    END;
    """
    )

    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())

except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

Bono del 20%: 38332.8


Crear una tabla llamada EMP, con similar estructura que la tabla EMPLOYEES, adicione a esta tabla una columna llamada STARS tipo VARCHAR2 de longitud 50.

In [21]:
try:
    cursor.execute(
    """
    CREATE TABLE emp AS SELECT * FROM employees
    """
    )
    cursor.execute("ALTER TABLE emp ADD stars VARCHAR2(50)")
except oracledb.Error as e:
    print(e)

ORA-00955: name is already used by an existing object
Help: https://docs.oracle.com/error-help/db/ora-00955/


Crear un bloque PL/SQL que permita guardar asteriscos en la columna STARS de la tabla EMP creada de acuerdo a lo siguiente:
a. Utilice el comando DEFINE para pasar el valor del ID del empleado
b. Inicialice la variable v_asterisk con NULL
c. Añada en la columna STARS del empleado un asterisco por cada $1000 del salario recibido, ejemplo:

In [22]:
try:
    p_id = 100
    cursor.execute(
    f"""
    DECLARE
        v_id emp.employee_id%TYPE := {p_id};
        v_asterisk VARCHAR2(50) := NULL;
        v_salario emp.salary%TYPE;
        v_contador NUMBER;
    BEGIN
        SELECT salary
        INTO v_salario
        FROM emp
        WHERE employee_id = v_id;
        v_contador := TRUNC(v_salario/1000);
        LOOP
            v_asterisk := v_asterisk || '*';
            v_contador := v_contador - 1;
        EXIT WHEN v_contador = 0;
        END LOOP;
        UPDATE emp
        SET stars = v_asterisk
        WHERE employee_id = v_id; 
        DBMS_OUTPUT.PUT_LINE(TO_CHAR(v_id) || ', ' || TO_CHAR(v_salario) || ', ' || v_asterisk);

    END;
    """
    )

    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())

except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

100, 31944, *******************************


las funciones se pueden usar como una columna en un select

In [23]:
try:
    p_id = 100
    cursor.execute(
    f"""
    CREATE OR REPLACE FUNCTION get_sal
    (p_dep_id IN departments.department_id%TYPE)
    RETURN NUMBER
    IS
        v_salary NUMBER :=0;
    BEGIN
        SELECT SUM(salary)
        INTO v_salary
        FROM employees
        WHERE department_id = p_dep_id;
        RETURN v_salary;
    END get_sal;
    """
    )

    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())

except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

In [24]:
try:
    p_id = 3
    cursor.execute(
    f"""
    DECLARE
        v_suma NUMBER;
        v_nombre departments.department_name%TYPE;
    BEGIN
        SELECT get_sal({p_id}), department_name
        INTO v_suma, v_nombre
        FROM departments
        WHERE department_id = {p_id};
        DBMS_OUTPUT.PUT_LINE(v_nombre || ': ' || TO_CHAR(v_suma));
    END;
    """
    )

    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())

except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

Purchasing: 24900


Crear un procedimiento llamado ADD_JOB para insertar un nuevo puesto de trabajo en
la tabla JOBS, ingrese el id y el puesto de trabajo mediante parámetros.
a. Ejecute el procedimiento ADD_JOB con el valor para jod_id igual a “IT_DBA” y
el job_title igual a “Database Administrator”


In [25]:
try:
    cursor.execute(
    f"""
    CREATE OR REPLACE PROCEDURE add_job
    (p_id IN jobs.job_id%TYPE, p_title IN jobs.job_title%TYPE)
    IS
    BEGIN
        INSERT INTO jobs
        (job_id, job_title)
        VALUES
        (p_id, p_title);
    END add_job;
    """
    )

    cursor.execute(
    f"""
    BEGIN
        add_job(22,'Database Administrator');
    END;
    """
    )

    cursor.execute(
    """
    SELECT * FROM jobs
    """
    )
    resultado = cursor.fetchall()
    for row in resultado:
        print(row)


except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

(1, 'Public Accountant', 4200.0, 9000.0)
(2, 'Accounting Manager', 8200.0, 16000.0)
(3, 'Administration Assistant', 3000.0, 6000.0)
(4, 'President', 20000.0, 40000.0)
(5, 'Administration Vice President', 15000.0, 30000.0)
(6, 'Accountant', 4200.0, 9000.0)
(7, 'Finance Manager', 8200.0, 16000.0)
(8, 'Human Resources Representative', 4000.0, 9000.0)
(9, 'Programmer', 4000.0, 10000.0)
(10, 'Marketing Manager', 9000.0, 15000.0)
(11, 'Marketing Representative', 4000.0, 9000.0)
(12, 'Public Relations Representative', 4500.0, 10500.0)
(13, 'Purchasing Clerk', 2500.0, 5500.0)
(14, 'Purchasing Manager', 8000.0, 15000.0)
(15, 'Sales Manager', 10000.0, 20000.0)
(16, 'Sales Representative', 6000.0, 12000.0)
(17, 'Shipping Clerk', 2500.0, 5500.0)
(18, 'Stock Clerk', 2000.0, 5000.0)
(19, 'Stock Manager', 5500.0, 8500.0)
(22, 'Database Administrator', None, None)
(20, 'Database Administrator', None, None)


Crear un procedimiento llamado DEL_JOB para eliminar un puestro de trabajo de la
tabla JOBS, crear la excepción necesaria cuando un puesto de trabajo no pueda ser
borrado o no exista en la base
a. Llame al procedimiento anterior para eliminar el puesto de trabajo con el
codigo IT_DBA

In [26]:
try:
    cursor.execute(
    f"""
    CREATE OR REPLACE PROCEDURE del_job
    (p_id IN jobs.job_id%TYPE)
    IS
    BEGIN
        DELETE FROM jobs
        WHERE job_id = p_id;
        IF SQL%ROWCOUNT > 0
        THEN DBMS_OUTPUT.PUT_LINE('Eliminado con exito');
        ELSE DBMS_OUTPUT.PUT_LINE('No se pudo eliminar');
        END IF;
    EXCEPTION
        WHEN OTHERS THEN
            DBMS_OUTPUT.PUT_LINE('Ocurrio un error al eliminar');
    END del_job;

    """
    )

    cursor.execute(
    f"""
    BEGIN
        del_job(22098);
    END;
    """
    )
    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())


except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

No se pudo eliminar


Crear un procedimiento llamado QUERY_EMP que recupere el salario y job_id de la tabla employees cuando envié como parámetro el código del empleado

In [27]:
try:
    cursor.execute(
    f"""
    CREATE OR REPLACE PROCEDURE query_emp
    (p_emp_id IN employees.employee_id%TYPE, p_salary OUT employees.salary%TYPE, p_job_id OUT employees.job_id%TYPE)
    IS
    BEGIN
        SELECT salary, job_id
        INTO p_salary, p_job_id
        FROM employees
        WHERE employee_id = p_emp_id;
        DBMS_OUTPUT.PUT_LINE('El empleado: ' || TO_CHAR(p_emp_id) || ' tiene un salario de: ' || TO_CHAR(p_salary) || ' y pertenece al job id: ' || TO_CHAR(p_job_id));
    END query_emp;

    """
    )

    cursor.execute(
    f"""
    DECLARE
        v_salario employees.salary%TYPE;
        v_job_id employees.job_id%TYPE;
    BEGIN
        query_emp(100, v_salario, v_job_id);
    END;
    """
    )
    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())


except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

El empleado: 100 tiene un salario de: 31944 y pertenece al job id: 4


Crear un procedimiento llamado QUERY_DEP que envíe como parámetro el código de un departamento y me devuelva información del número de empleados que existen y cual es el valor total de salarios pagados

In [28]:
try:
    cursor.execute(
    f"""
    CREATE OR REPLACE PROCEDURE query_dep
    (p_dep_id IN departments.department_id%TYPE, p_num_empleados OUT NUMBER, p_salarios_pagados OUT NUMBER)
    IS
    BEGIN
        SELECT COUNT(employee_id), SUM(salary)
        INTO p_num_empleados, p_salarios_pagados
        FROM employees
        WHERE department_id = p_dep_id;
        DBMS_OUTPUT.PUT_LINE('El departamento: ' || TO_CHAR(p_dep_id) || ' tiene total de ' || TO_CHAR(p_num_empleados) || ' empleados y ha pagado en salarios: ' || TO_CHAR(p_salarios_pagados));
    END query_dep;

    """
    )

    cursor.execute(
    f"""
    DECLARE
        v_total_emps NUMBER;
        v_total_salarios NUMBER;
    BEGIN
        query_dep(8, v_total_emps, v_total_salarios);
    END;
    """
    )
    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())


except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

El departamento: 8 tiene total de 6 empleados y ha pagado en salarios: 57700


Crear un procedimiento llamado QUERY_CIUDAD que envíe como parámetro el código de un departamento y me devuelva información del nombre de la ciudad y los empleados que trabajan en ese departamento

In [7]:
try:
    cursor.execute(
    f"""
    CREATE OR REPLACE PROCEDURE query_ciudad
    (p_dep_id IN departments.department_id%TYPE, p_ciudad OUT locations.city%TYPE, p_num_emp OUT NUMBER)
    IS
    BEGIN
        SELECT l.city, COUNT(e.employee_id)
        INTO p_ciudad, p_num_emp
        FROM departments d
        LEFT JOIN employees e ON d.department_id = e.department_id
        JOIN locations l ON l.location_id = d.location_id
        WHERE d.department_id = p_dep_id
        GROUP BY l.city;
        DBMS_OUTPUT.PUT_LINE('El departamento: ' || TO_CHAR(p_dep_id) || ' tiene total de ' || TO_CHAR(p_num_emp) || ' empleados y esta en la ciudad: ' || TO_CHAR(p_ciudad));
    END query_ciudad;

    """
    )

    cursor.execute(
    f"""
    DECLARE
        v_ciudad locations.city%TYPE;
        v_empleados NUMBER;
    BEGIN
        query_ciudad(5, v_ciudad, v_empleados);
    END;
    """
    )
    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())


except oracledb.Error as e:
    print(f"Ocurrió un error: {e}")

El departamento: 5 tiene total de 7 empleados y esta en la ciudad: South San Francisco
